In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MaskedLinear(nn.Module):
    def __init__(self, in_features, out_features, mask):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features)
        # Register the mask as a buffer (not trained)
        self.register_buffer('mask', mask) 

    def forward(self, x):
        # Apply mask to the weight matrix
        masked_weight = self.linear.weight * self.mask
        return F.linear(x, masked_weight, self.linear.bias)

def create_mask(input_dim, hidden_dim, output_dim, activation_fn=F.relu):
    # This function generates the necessary masks (M1, M2, M3) 
    # to ensure autoregressive property (e.g., in MADE paper)
    # The mask ensures output_i only depends on input_{<i}
    # ... (Implementation details needed here) ...
    pass 

# Example of a simple MADE network structure:
class MADE(nn.Module):
    def __init__(self, input_dim, hidden_dims):
        super().__init__()
        # ... use MaskedLinear layers here ...
        # (Must handle the masking logic carefully for 'm' and 'n' values)
        pass

In [ ]:
class MAFBlock(nn.Module):
    def __init__(self, input_dim, made_network):
        super().__init__()
        self.made = made_network
        # The MADE output size must be 2 * input_dim (for scale 's' and translation 't')

    def forward(self, x):
        # Forward pass (z = (x - t) / s): calculates log-likelihood
        s_and_t = self.made(x)
        s, t = s_and_t.chunk(2, dim=1)
        s = torch.sigmoid(s + 2) # Use sigmoid for stability
        
        z = (x - t) / s
        
        # Log-determinant of the Jacobian: log|det(dz/dx)| = -sum(log(s_i))
        log_det_J = -torch.sum(torch.log(s), dim=[1, 2, 3]) # Sum over feature dimensions
        return z, log_det_J

    def inverse(self, z):
        # Inverse pass (x = s * z + t): required for image generation (slow)
        # This requires an iterative (autoregressive) calculation:
        x = torch.zeros_like(z)
        # ... (Iterative calculation over dimensions needed here) ...
        # (This is the slow part mentioned in the prompt)
        return x

In [ ]:
class MAF(nn.Module):
    def __init__(self, num_blocks):
        super().__init__()
        self.blocks = nn.ModuleList([MAFBlock(...) for _ in range(num_blocks)])
        # Initialize base distribution (e.g., Standard Normal Gaussian)
        self.base_dist = torch.distributions.Normal(0, 1)

    def forward(self, x):
        # Calculate NLL (Negative Log Likelihood)
        log_prob = 0
        for block in self.blocks:
            x, log_det_J = block(x)
            log_prob += log_det_J
        
        # Add the log-probability of the final z under the base distribution
        log_prob += self.base_dist.log_prob(x).sum(dim=[1, 2, 3])
        return x, log_prob
        
    def generate(self, num_samples):
        # Sample z from base distribution
        z = self.base_dist.sample((num_samples, ...))
        # Pass through inverse transforms (in reverse order)
        for block in reversed(self.blocks):
            z = block.inverse(z) # This calls the slow inverse method
        return z